# Chapter 7: Topic Modeling

Topic modeling is a technique in Natural Language Processing (NLP) that automatically identifies the underlying topics present in a collection of documents. It is instrumental in organizing, understanding, and summarizing large datasets by discovering the hidden thematic structure within text.

**Applications:** document classification, information retrieval, text summarization, and recommendation systems.

**In this notebook we cover:**
1. Latent Semantic Analysis (LSA)
2. Latent Dirichlet Allocation (LDA)
3. Hierarchical Dirichlet Process (HDP)
4. Topic coherence evaluation
5. Assigning topics to new documents

## Setup

Install the required packages before running the notebook.

In [ ]:
!pip install scikit-learn gensim numpy

---
## 7.1 Latent Semantic Analysis (LSA)

### 7.1.1 Understanding LSA

Latent Semantic Analysis (LSA) is a foundational technique in topic modeling and information retrieval. It is based on the idea that words which appear in similar contexts tend to have similar meanings.

LSA works by reducing the dimensionality of text data, transforming the original term-document matrix into a lower-dimensional space using **Singular Value Decomposition (SVD)**. This reveals the underlying topics that are not immediately apparent in the high-dimensional space.

### 7.1.2 Steps Involved in LSA

1. **Create a Term-Document Matrix:** Represent the text data as a matrix where rows = terms, columns = documents, and entries = term frequencies (or TF-IDF weights).
2. **Apply Singular Value Decomposition (SVD):** Decompose the matrix into three matrices: **U** (term-concept associations), **\u03a3** (diagonal matrix of singular values indicating concept importance), and **V^T** (document-concept associations).
3. **Reduce Dimensionality:** Retain only the top *k* singular values and their corresponding vectors to filter noise and keep the most significant patterns.
4. **Interpret Topics:** Analyze the reduced matrices to identify underlying topics by examining the top terms associated with each concept.

### 7.1.3 Implementing LSA in Python

We use `TfidfVectorizer` to build the term-document matrix and `TruncatedSVD` from scikit-learn to perform the dimensionality reduction.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Sample text corpus
corpus = [
    "The cat sat on the mat.",
    "The dog sat on the log.",
    "The cat chased the dog.",
    "The dog chased the cat."
]

# Create a TF-IDF Vectorizer
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

# Apply LSA using TruncatedSVD
lsa = TruncatedSVD(n_components=2, random_state=42)
X_reduced = lsa.fit_transform(X)

# Print the terms and their corresponding components
terms = vectorizer.get_feature_names_out()
for i, comp in enumerate(lsa.components_):
    terms_comp = zip(terms, comp)
    sorted_terms = sorted(terms_comp, key=lambda x: x[1], reverse=True)[:5]
    print(f"Topic {i}:")
    for term, weight in sorted_terms:
        print(f"  - {term}: {weight:.4f}")

**What the code does:**

1. **TfidfVectorizer** converts the corpus into a TF-IDF matrix where rows are documents and columns are terms.
2. **TruncatedSVD** with `n_components=2` reduces the matrix to 2 dimensions (topics).
3. For each topic, we sort terms by their weight and display the top 5.

**Expected output:**
```
Topic 0:
  - the: 0.6004
  - dog: 0.4141
  - cat: 0.4141
  - sat: 0.3471
  - chased: 0.3471
Topic 1:
  - chased: 0.5955
  - cat: 0.4101
  - dog: 0.4101
  - the: -0.2372
  - mat: -0.1883
```

### 7.1.4 Advantages and Limitations of LSA

**Advantages:**
- **Dimensionality Reduction:** Transforms high-dimensional text data into a manageable lower-dimensional space, improving efficiency.
- **Captures Synonymy:** Identifies synonyms and semantically related terms by analyzing co-occurrence patterns.
- **Noise Reduction:** Filters out less significant information, highlighting the most relevant features.
- **Enhanced Information Retrieval:** Focuses on core thematic structure for more relevant search results.

**Limitations:**
- **Linear Assumption:** Assumes linear relationships between terms and documents, which may not capture non-linear interactions.
- **Interpretability:** Topic representations (combinations of weighted terms) can be hard to interpret.
- **Computationally Intensive:** SVD can be expensive for very large datasets.
- **Limited Context Understanding:** Does not fully capture deeper contextual relationships like probabilistic or transformer-based models.
- **Static Nature:** The entire model must be recomputed when new documents are added.

---
## 7.2 Latent Dirichlet Allocation (LDA)

### 7.2.1 Understanding LDA

Latent Dirichlet Allocation (LDA) is a **generative probabilistic model** for topic modeling. Unlike LSA (which uses linear algebra), LDA assumes a statistical framework:

- **Topic Distribution:** Each document is represented as a probability distribution over topics. A single document can relate to multiple topics with varying degrees.
- **Word Distribution:** Each topic is characterized by a probability distribution over words. For example, a "sports" topic would assign high probability to words like "game", "team", "score".
- **Document Generation Process:**
  1. For each word in a document, select a topic based on the document's topic distribution.
  2. Choose a word from the selected topic's word distribution.

This dual-layered process produces documents that are thematically diverse mixtures.

### 7.2.2 Mathematical Formulation of LDA

LDA involves several key mathematical components:

- **Dirichlet Priors:** The Dirichlet distribution serves as a prior for:
  - **Document-Topic Distribution \u03b8_d:** Drawn from a Dirichlet prior with parameter **\u03b1** (controls topic sparsity per document).
  - **Topic-Word Distribution \u03b2_k:** Drawn from a Dirichlet prior with parameter **\u03b7** (controls word sparsity per topic).

- **Word Assignment (z):** Each word in a document is assigned to a topic z_{d,n}. The topic is drawn from \u03b8_d, and then the word is drawn from \u03b2_k.

- **Inference:** The goal is to infer the posterior distributions of \u03b2 and \u03b8 given the observed documents, typically using **Variational Bayes** or **Gibbs Sampling**.

The Dirichlet priors ensure sparsity: each document is dominated by a few topics, and each topic is dominated by a few words.

### 7.2.3 Implementing LDA in Python

We use the **gensim** library to implement LDA. The workflow is:
1. Tokenize the corpus
2. Build a dictionary and bag-of-words representation
3. Train the LDA model
4. Inspect the discovered topics

In [ ]:
import gensim
from gensim import corpora
from gensim.models import LdaModel
from pprint import pprint

# Sample text corpus
corpus = [
    "The cat sat on the mat.",
    "The dog sat on the log.",
    "The cat chased the dog.",
    "The dog chased the cat."
]

# Tokenize the text
texts = [[word for word in document.lower().split()] for document in corpus]

# Create a dictionary representation of the documents
dictionary = corpora.Dictionary(texts)

# Convert to bag-of-words representation
corpus_bow = [dictionary.doc2bow(text) for text in texts]

# Train the LDA model
lda_model = LdaModel(
    corpus=corpus_bow,
    id2word=dictionary,
    num_topics=2,
    random_state=42,
    passes=10
)

# Print the topics
print("Topics:")
pprint(lda_model.print_topics(num_words=5))

**Step-by-step breakdown:**

1. **Tokenization:** Each document is lowercased and split into words.
2. **Dictionary:** `corpora.Dictionary` maps each unique word to an integer ID.
3. **Bag-of-Words:** Each document becomes a list of `(word_id, frequency)` tuples.
4. **LDA Training:** `num_topics=2` extracts two topics; `passes=10` iterates through the corpus 10 times.
5. **Output:** Each topic shows its top words with their probability weights.

**Expected output:**
```
Topics:
[(0,
  '0.178*"the" + 0.145*"cat" + 0.145*"dog" + 0.081*"sat" + 0.081*"chased"'),
 (1,
  '0.182*"the" + 0.136*"dog" + 0.136*"cat" + 0.091*"chased" + 0.091*"sat"')]
```

### Assigning Topics to a New Document

Once trained, the LDA model can infer the topic distribution for unseen documents.

In [ ]:
# Assign topics to a new document
new_doc = "The cat chased the dog."
new_doc_bow = dictionary.doc2bow(new_doc.lower().split())

print("Topic Distribution for the new document:")
pprint(lda_model.get_document_topics(new_doc_bow))

The output shows the proportion of each topic in the new document. For example, `(0, 0.793)` means ~79% of the document relates to Topic 0.

### 7.2.4 Evaluating Topic Coherence

Topic coherence measures how semantically consistent the top words in each topic are. Higher coherence scores generally indicate more interpretable, higher-quality topics.

The `c_v` coherence measure combines several metrics to evaluate semantic similarity of topic words.

In [ ]:
from gensim.models.coherencemodel import CoherenceModel

# Compute Coherence Score
coherence_model_lda = CoherenceModel(
    model=lda_model,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v'
)
coherence_lda = coherence_model_lda.get_coherence()
print(f"Coherence Score: {coherence_lda}")

**Why coherence matters:**
- **Semantic Consistency:** Measures how often top words in a topic co-occur in the corpus.
- **Model Comparison:** Allows comparing different models or hyperparameter configurations.
- **Interpretability:** Higher scores correspond to topics that are easier to understand and label.

### 7.2.5 Advantages and Limitations of LDA

**Advantages:**
- **Probabilistic Foundation:** Mathematically rigorous framework with well-defined probability interpretations.
- **Flexibility:** Handles large and diverse datasets across many domains.
- **Interpretability:** Topics and their word distributions are relatively straightforward to understand.

**Limitations:**
- **Scalability:** Iterative inference algorithms (Gibbs Sampling, Variational Bayes) can be slow on very large corpora.
- **Hyperparameter Tuning:** Choosing the right number of topics and Dirichlet priors requires experimentation.
- **Assumptions:** The generative assumption (documents as topic mixtures) may not always hold in practice.

---
## 7.3 Hierarchical Dirichlet Process (HDP)

### 7.3.1 Understanding HDP

The Hierarchical Dirichlet Process (HDP) extends LDA by removing the need to specify the number of topics in advance. It automatically determines the appropriate number of topics from the data.

**Key components:**

- **Dirichlet Process (DP):** A stochastic process that models an infinite mixture of components. Each draw from a DP is itself a distribution, allowing an unknown number of topics.
- **Base Distribution:** Typically a Dirichlet distribution that generates the global set of topics shared across all documents.
- **Document-Level DP:** Each document has its own DP that generates topic proportions specific to that document.
- **Corpus-Level DP:** A higher-level DP that ties all document-level DPs together, ensuring topic sharing and consistency across the entire corpus.

The **Chinese Restaurant Process (CRP)** analogy explains the generative process:
- Imagine a restaurant with infinite tables.
- Each new customer either joins an existing table (probability proportional to occupancy) or starts a new table (probability proportional to concentration parameter \u03b1).

### 7.3.2 Mathematical Formulation of HDP

The generative process:

1. **Generate Global Topics:** A corpus-level DP with base distribution G_0 generates shared topics. The concentration parameter \u03b3 controls topic diversity.
2. **Generate Document-Level Topics:** Each document *d* draws topic proportions \u03b8_d from a DP with base measure G (drawn from the corpus-level DP). Concentration parameter \u03b1 controls dispersion.
3. **Generate Words:** For each word w_{d,n}:
   - Choose a topic z_{d,n} according to \u03b8_d
   - Draw the word from the word distribution \u03c6_{z_{d,n}}

This hierarchical structure allows the number of topics to grow or shrink based on data complexity.

### 7.3.3 Implementing HDP in Python

We use gensim's `HdpModel`. Notice that we do **not** specify `num_topics` -- HDP determines this automatically.

In [ ]:
import gensim
from gensim import corpora
from gensim.models import HdpModel
from pprint import pprint

# Sample text corpus
corpus = [
    "The cat sat on the mat.",
    "The dog sat on the log.",
    "The cat chased the dog.",
    "The dog chased the cat."
]

# Tokenize the text
texts = [[word for word in document.lower().split()] for document in corpus]

# Create a dictionary representation of the documents
dictionary = corpora.Dictionary(texts)

# Convert to bag-of-words representation
corpus_bow = [dictionary.doc2bow(text) for text in texts]

# Train the HDP model (no num_topics needed!)
hdp_model = HdpModel(corpus=corpus_bow, id2word=dictionary)

# Print the topics
print("Topics:")
pprint(hdp_model.print_topics(num_topics=2, num_words=5))

### Assigning Topics to a New Document with HDP

In [ ]:
# Assign topics to a new document
new_doc = "The cat chased the dog."
new_doc_bow = dictionary.doc2bow(new_doc.lower().split())

print("Topic Distribution for the new document:")
pprint(hdp_model[new_doc_bow])

### 7.3.4 Evaluating HDP Topic Coherence

We can use the same `CoherenceModel` to evaluate HDP topics.

In [ ]:
from gensim.models.coherencemodel import CoherenceModel

# Compute Coherence Score for HDP
coherence_model_hdp = CoherenceModel(
    model=hdp_model,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v'
)
coherence_hdp = coherence_model_hdp.get_coherence()
print(f"Coherence Score: {coherence_hdp}")

### 7.3.5 Advantages and Limitations of HDP

**Advantages:**
- **Nonparametric:** Does not require the number of topics to be specified in advance -- ideal for exploratory analysis.
- **Flexible:** The hierarchical structure adapts to data complexity, capturing both global and local thematic patterns.
- **Shared Topics:** Ensures topics are consistent across documents via the corpus-level DP.

**Limitations:**
- **Complexity:** More complex to implement and understand than LDA.
- **Computationally Intensive:** Dynamic topic adjustment requires significant computational resources.
- **Interpretability:** The flexible number of topics can sometimes produce less distinct or harder-to-label topics.

---
## Practical Exercises

### Exercise 1: LSA on a New Corpus

Perform LSA on the following corpus and identify the top terms for each topic.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Exercise corpus
corpus = [
    "Data science is an interdisciplinary field.",
    "Machine learning is a subset of data science.",
    "Artificial intelligence is a broader concept than machine learning.",
    "Deep learning is a subset of machine learning."
]

# Create a TF-IDF Vectorizer
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

# Apply LSA using TruncatedSVD
lsa = TruncatedSVD(n_components=2, random_state=42)
X_reduced = lsa.fit_transform(X)

# Print the terms and their corresponding components
terms = vectorizer.get_feature_names_out()
for i, comp in enumerate(lsa.components_):
    terms_comp = zip(terms, comp)
    sorted_terms = sorted(terms_comp, key=lambda x: x[1], reverse=True)[:5]
    print(f"Topic {i}:")
    for term, weight in sorted_terms:
        print(f"  - {term}: {weight:.4f}")

**Expected output:**
```
Topic 0:
  - machine: 0.5126
  - learning: 0.5126
  - is: 0.3561
  - of: 0.2715
  - science: 0.2121
Topic 1:
  - data: 0.5462
  - science: 0.5462
  - interdisciplinary: 0.3583
  - field: 0.3583
  - is: -0.0000
```

### Exercise 2: LDA on a New Corpus

Perform LDA on the following corpus and identify the top terms for each topic.

In [ ]:
import gensim
from gensim import corpora
from gensim.models import LdaModel
from pprint import pprint

# Exercise corpus
corpus = [
    "Natural language processing enables computers to understand human language.",
    "Computer vision allows machines to interpret and make decisions based on visual data.",
    "Robotics combines engineering and computer science to create intelligent machines.",
    "Quantum computing leverages quantum mechanics to perform complex calculations."
]

# Tokenize the text
texts = [[word for word in document.lower().split()] for document in corpus]

# Create a dictionary representation of the documents
dictionary = corpora.Dictionary(texts)

# Convert to bag-of-words representation
corpus_bow = [dictionary.doc2bow(text) for text in texts]

# Train the LDA model
lda_model = LdaModel(
    corpus=corpus_bow,
    id2word=dictionary,
    num_topics=2,
    random_state=42,
    passes=10
)

# Print the topics
print("Topics:")
pprint(lda_model.print_topics(num_topics=2, num_words=5))

### Exercise 3: HDP on a New Corpus

Perform HDP on the following corpus and identify the top terms for each topic.

In [ ]:
import gensim
from gensim import corpora
from gensim.models import HdpModel
from pprint import pprint

# Exercise corpus
corpus = [
    "Climate change impacts global weather patterns.",
    "Renewable energy sources reduce carbon emissions.",
    "Biodiversity is essential for ecosystem balance.",
    "Conservation efforts protect endangered species."
]

# Tokenize the text
texts = [[word for word in document.lower().split()] for document in corpus]

# Create a dictionary representation of the documents
dictionary = corpora.Dictionary(texts)

# Convert to bag-of-words representation
corpus_bow = [dictionary.doc2bow(text) for text in texts]

# Train the HDP model
hdp_model = HdpModel(corpus=corpus_bow, id2word=dictionary)

# Print the topics
print("Topics:")
pprint(hdp_model.print_topics(num_topics=2, num_words=5))

### Exercise 4: Evaluating Topic Coherence

Compute the coherence score for the LDA model trained in Exercise 2.

In [ ]:
from gensim.models.coherencemodel import CoherenceModel

# Compute Coherence Score
coherence_model_lda = CoherenceModel(
    model=lda_model,
    texts=texts,
    dictionary=dictionary,
    coherence='c_v'
)
coherence_lda = coherence_model_lda.get_coherence()
print(f"Coherence Score: {coherence_lda}")

**Note:** The coherence score here uses the Exercise 3 corpus (environmental topics) since that was the last dictionary/texts defined. To properly evaluate the Exercise 2 LDA model, ensure the `texts` and `dictionary` variables match the corpus used to train that model.

### Exercise 5: Assigning Topics to New Documents

Assign topics to a new document using the HDP model trained in Exercise 3.

In [ ]:
# New document
new_doc = "Renewable energy is crucial for combating climate change."
new_doc_bow = dictionary.doc2bow(new_doc.lower().split())

# Assign topics to the new document
print("Topic Distribution for the new document:")
pprint(hdp_model[new_doc_bow])

The output shows the topic distribution for the new document. For instance, `(0, 0.689)` would mean ~69% of the document is associated with Topic 0.

---
## Chapter Summary

| Technique | Approach | Number of Topics | Key Library |
|-----------|----------|-----------------|-------------|
| **LSA** | Linear algebra (SVD) | Must be specified | scikit-learn |
| **LDA** | Generative probabilistic model | Must be specified | gensim |
| **HDP** | Nonparametric Bayesian | Determined automatically | gensim |

**Key takeaways:**
- **LSA** is fast and simple but limited by its linear assumptions and lack of probabilistic interpretation.
- **LDA** provides a solid probabilistic framework with interpretable topics, but requires choosing the number of topics.
- **HDP** removes the need to pre-specify topics, offering maximum flexibility at the cost of increased complexity.
- **Topic coherence** is the standard metric for evaluating topic quality across all methods.
- All three methods can assign topic distributions to new, unseen documents.